In [1]:
import pandas as pd
import mysql.connector
import numpy as np
import matplotlib.pyplot as plt
import re
from itertools import combinations
from datetime import datetime, timedelta

# Connect to Database

In [2]:

# MySQL Configuration - Replace these with your actual credentials
DB_NAME = "forex_db"
USER = "root"
PASSWORD = "Administrator$"
HOST = "localhost"

# Connect to MySQL
connection = mysql.connector.connect(
    host=HOST,
    user=USER,
    password=PASSWORD,
    database=DB_NAME
)
cursor = connection.cursor()

In [3]:
# Dict that stores fx pairs to its dataframe 
fx_tables_cache:dict[tuple[str, str], pd.DataFrame] = dict()

# Triangular Abr on FX 

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime, timedelta

def get_forex_data(table_name: str, start_date: str = None, end_date: str = None) -> pd.DataFrame:
    # Default to one month ago if start_date is not provided
    if start_date is None:
        start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S")

    # Default to now if end_date is not provided
    if end_date is None:
        end_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Extract currency pair from table name
    pattern = r"([A-Z]{6})"
    match = re.match(pattern, table_name)
    if not match:
        raise ValueError(f"Invalid table name format: {table_name}")
    
    currency_pair = match.group(1)
    currency1, currency2 = currency_pair[:3], currency_pair[3:]

    # Check cache first
    if (currency1, currency2, start_date, end_date) in fx_tables_cache:
        return fx_tables_cache[(currency1, currency2, start_date, end_date)]

    # SQL query with date filtering
    query = f"""
        SELECT timestamp, bid, ask 
        FROM {table_name} 
        WHERE timestamp BETWEEN '{start_date}' AND '{end_date}'
        ORDER BY timestamp
    """
    
    cursor.execute(query)
    rows = cursor.fetchall()
    
    # Convert to DataFrame
    df = pd.DataFrame(rows, columns=["timestamp", f"bid_{currency1}_{currency2}", f"ask_{currency1}_{currency2}"])
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    
    # Define original column names
    bid_col = f"bid_{currency1}_{currency2}"
    ask_col = f"ask_{currency1}_{currency2}"
    
    # Define reverse column names
    rev_bid_col = f"bid_{currency2}_{currency1}"
    rev_ask_col = f"ask_{currency2}_{currency1}"

    # Safely compute reverse prices using numpy to avoid division by zero
    df[rev_bid_col] = np.where(df[ask_col] != 0, 1 / df[ask_col], np.nan)
    df[rev_ask_col] = np.where(df[bid_col] != 0, 1 / df[bid_col], np.nan)

    # Cache the result
    fx_tables_cache[(currency1, currency2, start_date, end_date)] = df

    return df


In [5]:
usd_cad = get_forex_data('USDCAD_2024_12', '2024-12-27', '2024-12-30')
usd_cad

,timestamp,bid_USD_CAD,ask_USD_CAD,bid_CAD_USD,ask_CAD_USD
0,2024-12-27 00:00:00.602,1.44097,1.44106,0.693934,0.693977
1,2024-12-27 00:00:01.024,1.44097,1.44107,0.693929,0.693977
2,2024-12-27 00:00:02.118,1.44098,1.44107,0.693929,0.693972
3,2024-12-27 00:00:04.274,1.44099,1.44109,0.693919,0.693967
4,2024-12-27 00:00:04.603,1.44098,1.44107,0.693929,0.693972
...,...,...,...,...,...
81311,2024-12-29 23:59:16.980,1.44025,1.44038,0.694261,0.694324
81312,2024-12-29 23:59:32.047,1.44025,1.44037,0.694266,0.694324
81313,2024-12-29 23:59:34.781,1.44026,1.44034,0.694281,0.694319
81314,2024-12-29 23:59:34.984,1.44026,1.44037,0.694266,0.694319


In [6]:
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, coint

In [32]:
def join_forex(*dfs: list[pd.DataFrame]) -> pd.DataFrame:
    merged_df = dfs[0]
    for i in range(1, len(dfs)):
        merged_df = merged_df.merge(dfs[i], on='timestamp', how='inner')
    merged_df.replace(0, float("nan"))
    return merged_df

In [33]:
def is_stationary(series, p_value_threshold=0.05):
    result = adfuller(series.dropna())  # Drop NaN values before testing
    return result[1] < p_value_threshold  # Returns True if p-value < threshold

def find_cointegrated_pairs(*commodities: list[pd.DataFrame], p_value_threshold=0.05):
    cointegrated_pairs = []
    
    # Merge all commodity dataframes using join_forex function
    merged_data = join_forex(*commodities)
    
    # Extract all currency pairs
    currency_columns = [col for col in merged_data.columns if col != 'timestamp']
    
    # Iterate over all possible pairs of currency mid-prices
    for (asset1, asset2) in combinations(currency_columns, 2):
        # if the asset1 and asset2 contain the same currencies, skip
        currencies1 = asset1.split('_')[1:]
        currencies2 = asset2.split('_')[1:]
        if set(currencies1) == set(currencies2):
            continue

        series1 = merged_data[asset1].astype(float)
        series2 = merged_data[asset2].astype(float)

        # Run the Engle-Granger cointegration test
        _, p_value, _ = coint(series1, series2)
    
        if p_value < p_value_threshold:  # If cointegrated
            # Run OLS regression to find hedge ratio (beta)
            valid_idx = ~(series1.isna() | series2.isna())  # Get valid indices
            X = sm.add_constant(series1[valid_idx])
            y = series2[valid_idx]
            model = sm.OLS(y, X).fit()

            if series1.nunique() == 1:
                print(f"Skipping {asset1} - {asset2} due to constant values in series1")
                continue
            
            series1 = series1.replace(0, np.nan).dropna()
            series2 = series2.replace(0, np.nan).dropna()
            hedge_ratio = model.params[asset1]


            # Create the spread (linear combination)
            spread = series2 - hedge_ratio * series1

            # Check if the spread is stationary
            if is_stationary(spread):
                cointegrated_pairs.append((asset1, asset2, hedge_ratio, p_value))
                print(f"Cointegrated Pair Found: {asset1} - {asset2}, Hedge Ratio: {hedge_ratio:.3f}, p-value: {p_value:.5f}")
    
    # Convert results into a DataFrame
    return pd.DataFrame(cointegrated_pairs, columns=["Asset1", "Asset2", "Hedge Ratio", "P-Value"])

In [34]:
year = 2024
month = 12
start_date, end_date = f'{year}-{month}-27', f'{year}-{month}-30'

def get_currency_name(name: str):
    return f"{name}_{year}_{month}"

usd_cad = get_forex_data(get_currency_name("USDCAD"), start_date, end_date)
usd_jpy = get_forex_data(get_currency_name("USDJPY"), start_date, end_date)

In [35]:
# Find cointegrated pairs
cointegrated_pairs = find_cointegrated_pairs(usd_cad, usd_jpy)
cointegrated_pairs.head()

,Asset1,Asset2,Hedge Ratio,P-Value


In [ ]:
get_forex_data('CADJPY_2025_01', reverse=True)

,timestamp,bid_JPY_CAD,ask_JPY_CAD
0,2025-01-01 22:00:41.792,0.009172881293743177669537778512,0.009120925226654991882376548277
1,2025-01-01 22:00:50.262,0.009172881293743177669537778512,0.009121008418690770451581126809
2,2025-01-01 22:01:01.033,0.009172713013327952008365514268,0.009121091612244153380276551498
3,2025-01-01 22:01:11.018,0.009172628875435699871583195744,0.009121174807315182195466776121
4,2025-01-01 22:01:31.035,0.009172544739086964896671283514,0.009121258003903898425670868526
...,...,...,...
9890067,2025-01-31 21:59:56.039,0.009569561139926122987999770331,0.009272739074395185593872574020
9890068,2025-01-31 21:59:56.086,0.009377784029633797533642800206,0.009351912466099317310389974750
9890069,2025-01-31 21:59:56.179,0.009377871973291820620064894874,0.009361192241443870291320302554
9890070,2025-01-31 21:59:56.383,0.009377608147265958344664609845,0.009361104610344020594430142757


In [ ]:
fx_tables, pairs = getForexTables()
fx_tables, pairs

({'AUDJPY': ('AUDJPY_2025_01', False),
  'JPYAUD': ('AUDJPY_2025_01', True),
  'AUDNZD': ('AUDNZD_2025_01', False),
  'NZDAUD': ('AUDNZD_2025_01', True),
  'AUDUSD': ('AUDUSD_2025_01', False),
  'USDAUD': ('AUDUSD_2025_01', True),
  'CADJPY': ('CADJPY_2025_01', False),
  'JPYCAD': ('CADJPY_2025_01', True),
  'CHFJPY': ('CHFJPY_2025_01', False),
  'JPYCHF': ('CHFJPY_2025_01', True),
  'EURCHF': ('EURCHF_2025_01', False),
  'CHFEUR': ('EURCHF_2025_01', True),
  'EURGBP': ('EURGBP_2025_01', False),
  'GBPEUR': ('EURGBP_2025_01', True),
  'EURJPY': ('EURJPY_2025_01', False),
  'JPYEUR': ('EURJPY_2025_01', True),
  'EURPLN': ('EURPLN_2025_01', False),
  'PLNEUR': ('EURPLN_2025_01', True),
  'EURUSD': ('EURUSD_2025_01', False),
  'USDEUR': ('EURUSD_2025_01', True),
  'GBPJPY': ('GBPJPY_2025_01', False),
  'JPYGBP': ('GBPJPY_2025_01', True),
  'GBPUSD': ('GBPUSD_2025_01', False),
  'USDGBP': ('GBPUSD_2025_01', True),
  'NZDUSD': ('NZDUSD_2025_01', False),
  'USDNZD': ('NZDUSD_2025_01', True),

In [ ]:
fx_triplets = getTriangularTriplets(pairs)

In [ ]:
fx1, fx2, fx3 = fx_triplets[0]
fxdata = join_forex_data(fx1, fx2, fx3)
fxdata

OperationalError: 2013 (HY000): Lost connection to MySQL server during query